# Tool Calling Fine-Tuning: Full Fine-Tuning on T4 GPU
Bu notebook, **Qwen2.5-0.5B** modeli uzerinde **Full Fine-Tuning (Tum Parametreler Egitilebilir, 16-bit float16)** yontemiyle Tool / Function Calling yetenegini egitmek ve degerlendirmek icin hazirlanmistir.

- **Metot:** Full Fine-Tuning (494 Milyon Parametre - %100 Trainable)
- **Taban Model:** Qwen/Qwen2.5-0.5B (16-bit float16)
- **Hedef Donanim:** Google Colab T4 (16GB VRAM, float16)
- **Sekans Uzunlugu:** max_seq_len = 2048
- **Bellek Optimizasyonu:** `batch_size: 1`, `grad_accum_steps: 8` (efektif batch = 8), Gradient Checkpointing aktif.

---

### 1. GPU ve Donanim Kontrolu

In [ ]:
!nvidia-smi

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"bf16 support: {torch.cuda.is_bf16_supported()}")

### 2. Projeyi Klonla ve Guncelle

In [ ]:
import os

REPO_URL = "https://github.com/fatihkadim/tool-calling-ft.git"
PROJECT_DIR = "tool-calling-ft"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}

%cd {PROJECT_DIR}
!git pull origin main
!pwd

### 3. Bagimliliklarin Kurulumu

In [ ]:
!pip install -q --upgrade pip
!pip install -q "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "datasets>=3.0" "trl>=0.11" "accelerate>=1.0" pyyaml tqdm pandas matplotlib

# uv_build backend'ini kur, sonra projeyi editable olarak yukle
!pip install -q "uv_build>=0.11.7,<0.12.0"
!pip install -q --no-build-isolation -e .

### 4. Veri Setini Hazirla (Gerekirse)

In [ ]:
!python -m tool_calling_ft.data.prepare_dataset

### 5. T4 Icin Full Fine-Tuning Config Olustur

> **Dikkat:** Full FT'de 494M model parametresinin tamami ve AdamW optimizer durumlari (momentum + variance) bellekte tutulur.
> T4'un 16GB VRAM sinirina sigmasi icin `batch_size: 1`, `grad_accum_steps: 8` ve Gradient Checkpointing kullanilir.

In [ ]:
import yaml

config = {
    "method": "full_ft",
    "base_model": "Qwen/Qwen2.5-0.5B",
    "dataset": "NousResearch/hermes-function-calling-v1",
    "output_dir": "checkpoints/full_ft",
    "training": {
        "epochs": 3,
        "batch_size": 1,
        "grad_accum_steps": 8,
        "learning_rate": 2e-5,  # Full FT'de daha dusuk LR kullanilir
        "max_seq_len": 2048,
        "warmup_ratio": 0.05,
        "save_steps": 200,
        "seed": 42,
    },
}

with open("configs/full_ft_t4.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("T4 Full FT config yazildi: configs/full_ft_t4.yaml")
print(yaml.dump(config, default_flow_style=False))

### 6. Full Fine-Tuning Egitimini Baslat

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.training.train --config configs/full_ft_t4.yaml

### 7. Canli Demo (Inference Testi)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "checkpoints/full_ft"

tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

system_prompt = """You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags.
<tools>
[{"type": "function", "function": {"name": "get_current_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}}]
</tools>
For each function call return a json object with function name and arguments within <tool_call> </tool_call> tags."""

user_query = "What is the weather in Tokyo in celsius?"
prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_ids = [tokenizer.eos_token_id]
if isinstance(im_end_id, int) and im_end_id != tokenizer.eos_token_id:
    stop_ids.append(im_end_id)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.1,
        eos_token_id=stop_ids,
    )

response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("=" * 50)
print("Full FT MODEL CIKTISI:")
print(response.strip())
print("=" * 50)

### 8. Full FT Degerlendirme (Evaluation)

In [ ]:
# Full FT Modelini Degerlendir (Adapter yok, dogrudan model path verilir):
!CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.eval.harness \
    --method full_ft \
    --model checkpoints/full_ft \
    --dataset data/processed/eval_subset.jsonl

### 9. Sonuclari Yedekle (Google Drive)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/tool_calling_full_ft_results
!cp -r checkpoints/full_ft /content/drive/MyDrive/tool_calling_full_ft_results/ 2>/dev/null || echo 'checkpoints klasoru yok'
!cp -r reports /content/drive/MyDrive/tool_calling_full_ft_results/ 2>/dev/null || echo 'reports klasoru yok'
print("Full FT Checkpoint ve raporlar Drive'a kaydedildi!")